# Chạy thử pipeline RAG – Quy chế đào tạo SGU
Cần Ollama đang chạy (`ollama serve`) và đã pull model trong `src/config.py` (profile `local`: `qwen3:4b`).

In [2]:
import os, sys, logging
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "eval"))

os.environ["RAG_PROFILE"] = "local"   # đổi thành "server" nếu dùng server Ollama từ xa
logging.getLogger("httpx").setLevel(logging.WARNING)

from config import get_config
cfg = get_config()
print(cfg)

RAGConfig(embed_model_name='bkai-foundation-models/vietnamese-bi-encoder', embed_device='cuda', llm_model_name='qwen3:4b', ollama_base_url='http://localhost:11434', request_timeout=180.0, reranker_model_name=None, reranker_top_n=3, similarity_top_k=10, response_mode='compact', streaming=True, persist_dir='storage/vector_index', corpus_sources=[('data/raw/quy_che_dao_tao_2021.pdf', 'Quy chế đào tạo 2021'), ('data/raw/qd_sua_doi_2025.pdf', 'QĐ sửa đổi 2025')])


## 0. Khởi động Ollama (bỏ qua nếu đã chạy)
Chỉ dùng được với profile `local`. Profile `server` dùng Ollama từ xa.

In [3]:
import subprocess, time, urllib.request

def ollama_up() -> bool:
    try:
        urllib.request.urlopen(cfg.ollama_base_url + "/api/tags", timeout=2)
        return True
    except Exception:
        return False

ollama_proc = None
if ollama_up():
    print("Ollama đã chạy sẵn.")
else:
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
    print("Ollama đã khởi động." if ollama_up() else "Không khởi động được Ollama.")

# Khi xong, tắt tiến trình do notebook mở (chạy ở cell cuối nếu muốn):
# ollama_proc and ollama_proc.terminate()

Ollama đã khởi động.


## 1. Node parser: tách theo Điều

In [4]:
from index_builder import load_documents
from node_parser import parse_by_dieu, validate_nodes

nodes = parse_by_dieu(load_documents())
print(validate_nodes(nodes))
for n in nodes[:3]:
    print(n.metadata["dieu"], "|", n.text[:100].replace("\n", " "))

/mnt/data/1_Projects/sgu-rag-llamaindex/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'tong_so_node': 21, 'node_khong_ro_dieu': 0, 'ty_le_khong_ro': '0.0%', 'do_dai_trung_binh_ky_tu': 1538}
1 | Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng  I. Văn bản này cụ thể hóa Quy chế đào tạo trình độ 
2 | Điều 2. Chương trình đào tạo và thời gian học tập  1, Chương trình đào tạo được xây dựng theo đơn vị
3 | Điều 3. Phương thức tổ chức đào tạo  I. Nhà trường tổ chức đào tạo theo tín chỉ đối với chương trình


## 2. Build / load Index

In [5]:
from index_builder import build_or_load_index

index = build_or_load_index(cfg)   # rebuild=True để build lại
print("Số node trong Index:", len(index.docstore.docs))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1101.11it/s]


Số node trong Index: 21


## 3. Retriever

In [6]:
from retriever import get_retriever

retriever = get_retriever(index, cfg)
question = "Điểm trung bình bao nhiêu thì được xếp loại Giỏi?"
for i, n in enumerate(retriever.retrieve(question), 1):
    print(f"[{i}] score={n.score:.4f} | Điều {n.node.metadata['dieu']} | {n.node.text[:90]!r}")

[1] score=0.2472 | Điều 13 | 'Điều 13. Công nhận tốt nghiệp và cấp bằng tốt nghiệp\n\n1. Người học được xét và công nhận t'
[2] score=0.2418 | Điều 10 | 'Điều 10. Đánh giá kết quả học tập theo học kì, nắm học\n\n1. Dựa trên kết quả đánh giá các h'
[3] score=0.1663 | Điều 9 | 'Điều 9. Đánh giá và tính điểm học phần\n\n1. Đối với mỗi học phần, người học được đánh giá q'
[4] score=0.1657 | Điều 19 | 'Điều 19. Xử lý vi phạm đối với người học\n\n1. Người học có gian lận trong thi, kiểm tra, đá'
[5] score=0.1571 | Điều 11 | 'Điều 11. Xử lý kết quả học tập theo tín chỉ\n\n1. Cuối mỗi học kì chính, người học bị cảnh b'
[6] score=0.1171 | Điều 18 | 'Điều 18. Học liên thông đối với người có văn bằng khác\n\n1. Người đã tốt nghiệp các trình đ'
[7] score=0.1024 | Điều 3 | 'Điều 3. Phương thức tổ chức đào tạo\n\nI. Nhà trường tổ chức đào tạo theo tín chỉ đối với ch'
[8] score=0.0816 | Điều 17 | 'Điều 17. Học cùng lúc hai chương trình\n\n1. Người học theo phương thức đào tạo tín chỉ có t'
[9] score=

## 4. Query engine (gọi LLM qua Ollama)

In [7]:
from query_engine import build_query_engine, ask

engine = build_query_engine(index, cfg)
ask(engine, "Một năm học có bao nhiêu học kỳ?", cfg)

Okay, let's see. The user is asking how many semesters a year has based on the provided context. The original answer says that according to the University of Saigon's 2021 education regulation, a year has 3 semesters: 2 main and 1 auxiliary.

Wait, the new context given here is in Vietnamese. The new context starts with "đào tạo;" followed by some text about training reporting to the provincial people's committee, storage of educational materials, and then a section on Điều 12 about recognizing academic results and credit transfer. The user's query is in Vietnamese: "Một năm học có bao nhiêu học kỳ?" which translates to "How many semesters does a year have?"

Hmm, but the original answer was based on Điều 6, khoản 2 which says "Mỗi năm học có 02 học kì chính với tổng số tối thiểu 30 tuần lên lớp và 01 học kì phụ với tối thiểu 05 tuần lên lớp." So that's 2 main semesters and 1 auxiliary, totaling 3 semesters.

But the new context provided here doesn't include Điều 6. The new context giv

Thử câu hỏi khác:

In [8]:
ask(engine, "Sinh viên bị buộc thôi học trong những trường hợp nào?", cfg)

Okay, let's see. I need to check if the new context provided is useful for refining the original answer. The user gave a new context in Vietnamese related to education regulations, specifically about learning, maintaining discipline, and quality education. The original answer was in Vietnamese about the cases where students are expelled.

The new context includes sections on internal auditing, online teaching, and responsibilities of different departments. But the original answer was about expulsion cases, which were detailed in the regulations (Điều 11 and Điều 19). The new context here seems to focus on other aspects like online learning, quality assurance, internal supervision, etc., but doesn't mention any new expulsion cases. The original answer already lists four specific cases from the regulations (Điều 11 and Điều 19). Since the new context doesn't add any new information about expulsion cases, I should repeat the original answer.

The instruction says: if the new context isn't

## 5. Đánh giá Recall@k

In [9]:
import json
from evaluate import evaluate, QUESTIONS_FILE

questions = json.loads(QUESTIONS_FILE.read_text(encoding="utf-8"))
recall = evaluate(questions, retriever)
for k, v in recall.items():
    print(f"Recall@{k}: {v:.1%}")

Recall@1: 64.5%
Recall@3: 87.1%
Recall@5: 90.3%
Recall@10: 100.0%
